<a href="https://colab.research.google.com/github/DanielML84/Big-Data/blob/main/ejercicio_streaming_daniel_melendez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS (Herramientas del sistema)
# ==============================================================================
import time            # Para medir tiempos reales y pausar la ejecución (time.sleep)
import random          # Para simular el azar y generar tráfico aleatorio
from datetime import datetime  # Para convertir los números de tiempo en horas legibles
from collections import defaultdict  # Diccionario especial que crea listas automáticas

# ==============================================================================
# 1. SIMULADOR DEL STREAMING DE RED (Fábrica que genera los clics/logs)
# ==============================================================================
class SistemaStreaming:
    """Clase encargada de inventar las acciones de los usuarios en tiempo real."""

    def __init__(self):
        """Método constructor: se ejecuta automáticamente al crear el simulador."""
        # Creamos una lista de strings con los nombres de los usuarios ficticios
        self.usuarios = ["user_A", "user_B", "user_C", "hacker_99", "user_D"]

        # Creamos una lista con las secciones (endpoints) de la web que van a visitar
        self.endpoints = ["/login", "/dashboard", "/profile", "/api/data"]

    def simular_trafico(self, n_eventos=30):
        """Generador (usa 'yield') que fabrica un total de 'n_eventos' uno por uno."""

        # Bucle que se repetirá la cantidad de veces que le hayamos pedido (ej. 25 veces)
        # Usamos '_' porque no necesitamos usar el número de la vuelta actual aquí dentro
        for _ in range(n_eventos):

            # SIMULACIÓN DE AMBIENTE: Forzamos que haya un atacante activo en la red
            # random.random() da un número entre 0.0 y 1.0. Si es menor a 0.4 (40% de probabilidad):
            if random.random() < 0.4:
                usuario = "hacker_99"  # Elegimos obligatoriamente al hacker
                endpoint = "/login"    # Lo mandamos directo a intentar romper la contraseña

            # El otro 60% de las veces simulamos el comportamiento de usuarios normales:
            else:
                usuario = random.choice(self.usuarios)    # Escoge un usuario al azar de la lista
                endpoint = random.choice(self.endpoints)  # Escoge una sección web al azar

            # CONSTRUCCIÓN DEL LOG: Guardamos los datos de este clic en un diccionario
            evento = {
                'usuario': usuario,        # Guarda el string del usuario seleccionado
                'timestamp': time.time(),  # Guarda la hora exacta actual en segundos (ej: 1716382.45)
                'endpoint': endpoint       # Guarda el string de la página que visitó
            }

            # LA MAGIA DEL STREAMING: Enviamos el evento hacia afuera sin destruir la función
            # 'yield' pausa esta función, entrega el diccionario 'evento' al bucle for de afuera,
            # y se queda esperando a que le vuelvan a pedir otro elemento para reanudarse.
            yield evento

            # Freno de tiempo para simular la realidad: el programa espera un instante al azar
            # de entre 0.1 y 0.4 segundos antes de pasar a la siguiente vuelta del bucle.
            time.sleep(random.uniform(0.1, 0.4))


# ==============================================================================
# 2. DETECTOR DE FRAUDE (El WAF / Muro de Seguridad)
# ==============================================================================
class FiltroWAF:
    """Clase que analiza cada log y decide si el usuario es legal o un peligro."""

    def __init__(self, max_logins=3, ventana_tiempo=2.0):
        """Constructor: Guarda las reglas de baneo que le asigne el profesor."""
        self.max_logins = max_logins          # Límite: Cuántos inicios de sesión dejamos intentar
        self.ventana_tiempo = ventana_tiempo  # Tiempo: En qué margen de segundos medimos el límite

        # El archivador automático: si un usuario no existía, le crea una lista vacía [] al llamarlo
        # Guardará una estructura como: {"user_A": [tiempo1, tiempo2], "hacker_99": [tiempo1]}
        self.historial_logins = defaultdict(list)

        # Usamos un 'set' (conjunto). Es como una lista, pero ideal para buscar dentro muy rápido.
        # Guardará los nombres de los usuarios bloqueados definitivamente: {"hacker_99"}
        self.usuarios_baneados = set()

    def procesar_evento(self, evento):
        """Recibe un diccionario 'evento', aplica las reglas y devuelve un diagnóstico (string)."""

        # Extraemos los datos del diccionario para trabajar de forma más cómoda:
        usuario = evento['usuario']    # Quién realiza la acción
        endpoint = evento['endpoint']  # A dónde intenta entrar
        ahora = evento['timestamp']    # El segundo exacto en el que ocurrió

        # REGLA 1: FILTRO DE LISTA NEGRA
        # Revisamos si el nombre del usuario ya se encuentra dentro de nuestro conjunto de baneados
        if usuario in self.usuarios_baneados:
            return "BLOQUEADO (Usuario Baneado)" # Detiene el análisis y bloquea el acceso en el acto

        # REGLA 2: CONTROL DE INTENTOS DE LOGIN
        # Si el usuario no estaba baneado de antes, pero vemos que su acción es entrar a "/login":
        if endpoint == "/login":

            # Buscamos la lista de este usuario en el historial y le añadimos (.append) el tiempo actual
            self.historial_logins[usuario].append(ahora)

            # LIMPIEZA DE HISTORIAL (Comprensión de listas):
            # Recorremos todos los tiempos 't' que tenía guardados ese usuario. Restamos 'ahora - t'.
            # Si el resultado es menor o igual a 'ventana_tiempo' (2 segundos), significa que es reciente.
            # Los que tengan más de 2 segundos de antigüedad se eliminan automáticamente de la lista.
            self.historial_logins[usuario] = [
                t for t in self.historial_logins[usuario]
                if ahora - t <= self.ventana_tiempo
            ]

            # REGLA 3: VERIFICACIÓN DE FRAGMENTO DE FRAUDE
            # Medimos cuántos elementos (len) quedaron en la lista tras haber borrado los viejos.
            # Si el número de intentos recientes supera el máximo permitido (ej: más de 3 intentos):
            if len(self.historial_logins[usuario]) > self.max_logins:

                # Añadimos al usuario al conjunto de la muerte (lista negra)
                self.usuarios_baneados.add(usuario)

                # Devolvemos la alerta roja para mostrar en la consola
                return "🚨 ¡BANEADO POR ATAQUE DE FUERZA BRUTA!"

        # Si el usuario superó todas las pruebas (no está baneado ni saturó el login):
        return "PERMITIDO"


# ==============================================================================
# 3. EJECUCIÓN PRINCIPAL (El motor que une el Stream con el WAF)
# ==============================================================================
# Imprimimos un encabezado en la pantalla de la consola
print("=== INICIANDO SIMULACIÓN DE WAF EN STREAMING ===")

# Instanciamos (creamos) los objetos reales a partir de los planos de nuestras clases:
stream = SistemaStreaming()  # Iniciamos nuestra fábrica de datos en vivo

# Creamos nuestro WAF con la regla estricta: Máximo 3 logins en un lapso de 2.0 segundos
waf = FiltroWAF(max_logins=3, ventana_tiempo=2.0)

# CONSUMO EN VIVO: Iniciamos el bucle que lee el flujo del streaming
# - stream.simular_trafico(n_eventos=25): Activa el generador para darnos 25 eventos.
# - enumerate(..., start=1): Cuenta las vueltas del bucle empezando desde el número 1.
# - En cada vuelta, 'i' recibe el número de evento y 'evento' recibe el diccionario con los datos.
for i, evento in enumerate(stream.simular_trafico(n_eventos=25), start=1):

    # PASO CLAVE: Enviamos el diccionario actual al WAF y guardamos lo que nos responda
    # Nos devolverá uno de estos textos: "PERMITIDO", "BLOQUEADO (...)" o "🚨 ¡BANEADO!"
    resultado_waf = waf.procesar_evento(evento)

    # FORMATEO VISUAL: Convertimos el timestamp numérico a un reloj comprensible
    # - datetime.fromtimestamp(...): Convierte el float de segundos en un objeto de fecha y hora.
    # - strftime('%H:%M:%S.%f'): Lo transforma en texto con formato Hora:Minuto:Segundo.Microsegundos.
    # - [:-3]: Corta las últimas 3 cifras de los microsegundos para dejar solo milisegundos limpios.
    hora_legible = datetime.fromtimestamp(evento['timestamp']).strftime('%H:%M:%S.%f')[:-3]

    # IMPRESIÓN DEL LOG FINAL EN LA CONSOLA
    # Usamos f-strings estructurados:
    # - {i:02d}: Asegura que el número de evento ocupe 2 dígitos (ej: '01', '02', '10')
    # - {evento['usuario']:<10}: Alinea el nombre a la izquierda rellenando con espacios hasta 10 caracteres
    # - {evento['endpoint']:<10}: Hace lo mismo con las rutas para que toda la pantalla quede tabulada y ordenada
    print(f"[{hora_legible}] Evento {i:02d} | Usuario: {evento['usuario']:<10} | Endpoint: {evento['endpoint']:<10} -> Estado WAF: {resultado_waf}")


=== INICIANDO SIMULACIÓN DE WAF EN STREAMING ===
[12:11:32.313] Evento 01 | Usuario: hacker_99  | Endpoint: /login     -> Estado WAF: PERMITIDO
[12:11:32.540] Evento 02 | Usuario: user_A     | Endpoint: /profile   -> Estado WAF: PERMITIDO
[12:11:32.673] Evento 03 | Usuario: user_D     | Endpoint: /dashboard -> Estado WAF: PERMITIDO
[12:11:32.888] Evento 04 | Usuario: user_A     | Endpoint: /profile   -> Estado WAF: PERMITIDO
[12:11:33.138] Evento 05 | Usuario: user_C     | Endpoint: /profile   -> Estado WAF: PERMITIDO
[12:11:33.296] Evento 06 | Usuario: user_C     | Endpoint: /login     -> Estado WAF: PERMITIDO
[12:11:33.505] Evento 07 | Usuario: hacker_99  | Endpoint: /login     -> Estado WAF: PERMITIDO
[12:11:33.647] Evento 08 | Usuario: hacker_99  | Endpoint: /login     -> Estado WAF: PERMITIDO
[12:11:33.775] Evento 09 | Usuario: hacker_99  | Endpoint: /dashboard -> Estado WAF: PERMITIDO
[12:11:33.887] Evento 10 | Usuario: user_A     | Endpoint: /login     -> Estado WAF: PERMITIDO
[